<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi-e2/notebooks/gsm8k/Experiment_2_Dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Experiment 2: Two-Step Lookahead MEDAL vs. AB-MEDAL on GSM8K — Dev (5 Questions)

This notebook evaluates **Experiment 2 (E2)** comparing **MEDAL** and **AB-MEDAL** on GSM8K using a **Two-Step Lookahead Fractional Information Gain (FIG)** reward function with `LLaDA-8B-Instruct`.

### Mathematical Formulation:
Rather than computing immediate 1-step information gain, for each candidate child state $s_i$, search projects the highest-confidence action $a^*_i \in A(s_i)$ to form grandchild $g_i = s_i \oplus a^*_i$, computing:
$$
R_{\mathrm{FIG}}(s, s_i) = \frac{H(s) - H(g_i)}{\max(H(s), \epsilon)}
$$

- **MEDAL (Two-Step Ahead)**: Expands all $K_2$ child actions at once; for each child $s_i$, evaluates $s_i$ and its greedy grandchild $g_i$, caching both states and computing $R_{\mathrm{FIG}}(s, s_i)$. Backpropagates mean backed-up rewards.
- **AB-MEDAL (Two-Step Ahead)**: When selecting `GEN`, expands the top remaining action from $s \to s_i$, then greedily takes the top action from $s_i \to g_i$, evaluates both forwards, updates Student-$t$ Gaussian posteriors with $R_{\mathrm{FIG}}$, and backpropagates.
- **State Evaluation Caching**: Grandchild analyses are cached in `cached_analyses` so that subsequent tree expansions incur **0 redundant NFEs**.


## 1. Verify GPU Hardware
Verify that a CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM) is active.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name} | Compute Capability: {gpu.major}.{gpu.minor} | Total VRAM: {vram_gb:.2f} GiB')
if vram_gb < 20.0:
    print('Warning: VRAM is below 22 GiB. If OOM occurs, use --batch-size 1.')
else:
    print('VRAM check passed.')


## 2. Repository Setup
Clone or update the GitHub repository on branch `adi`.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/CooL-Legend/CSCI544-DLM.git'
BRANCH = 'adi-e2'
REPO_DIR = Path('/content/CSCI544-DLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        token = os.environ.get('GITHUB_TOKEN', '')
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                pass
        clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
        subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Working directory: {Path.cwd()}')


## 3. Install Dependencies
Install pinned dependencies and install editable package.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/llada-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
    import importlib
    importlib.invalidate_caches()
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
print('Dependencies installed successfully.')


## 4. Run Algorithmic Pre-Flight Unit Tests
Run unit tests to verify 2-step lookahead FIG reward computation, state caching, and batched parity.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/tasks/test_experiment_2.py', '-v'],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Pre-flight E2 unit tests failed!')
print('All E2 algorithmic unit tests passed.')


## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent artifact and model caching.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/CSCI544-DLM')
OUTPUT_DIR = Path('results/e2_dev')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_ROOT / 'results/e2_dev'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = DRIVE_ROOT / 'cache/huggingface'
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

print(f'Output directory: {OUTPUT_DIR.resolve()}')


## 6. Experiment Controls & Configuration
Configure 5-question frozen `dev` set, matched search budgets [256, 512], and cooperative batch execution.

In [ ]:
from dlm_project.tasks.gsm8k.experiment_2 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context
)

STAGE = 'dev'
METHODS = ('medal', 'ab_medal')
BUDGETS = (256, 512)
BATCH_SIZE = 2
RESUME = True

ctx = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
)
print(f'Stage: {ctx["stage"]} | Questions: {len(ctx["examples"])} | Budgets: {BUDGETS} | Methods: {METHODS}')


## 7. Cache & Load LLaDA-8B-Instruct
Load `LLaDA-8B-Instruct` model and tokenizer on CUDA in bfloat16, and prepare tokenized evaluation prompts.

In [ ]:
from dlm_project.tasks.gsm8k.experiment_2 import load_model, prepare

print('Loading model and tokenizer on GPU in bfloat16...')
wrapper = load_model(ctx, device='cuda')
prepared = prepare(wrapper, ctx)
print(f'Prepared {len(prepared)} tokenized evaluation prompts.')


## 8. Execute Experiment 2 (Dev (5 Questions))
Execute MEDAL and AB-MEDAL with the 2-step lookahead FIG reward across budgets 256 and 512.

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_2 import run_e2_stage

print(f'Starting E2 execution on {len(ctx["examples"])} questions for {METHODS} at budgets {BUDGETS}...')
t_start = perf_counter()
q_rows, agg_rows = run_e2_stage(
    wrapper,
    ctx,
    OUTPUT_DIR,
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
total_elapsed = perf_counter() - t_start
print(f'\nE2 Dev execution complete in {total_elapsed / 60:.2f} minutes ({len(q_rows)} checkpoint outcomes).')


## 9. Experiment 2 Results & Paired Comparison
Display aggregate accuracy and paired comparison (McNemar exact test) at matched budgets.

In [ ]:
import pandas as pd
from IPython.display import display
from scripts.run_e2 import compute_paired_comparison

agg_file = OUTPUT_DIR / 'summaries' / f'e2_{STAGE}_aggregate.csv'
if agg_file.is_file():
    df_agg = pd.read_csv(agg_file)
    print('=== E2 Dev Aggregate Metrics ===')
    display(df_agg[['method', 'search_budget', 'sample_count', 'accuracy_percent', 'ci_95_low', 'ci_95_high', 'mean_total_nfe', 'mean_selected_depth']])

paired = compute_paired_comparison(q_rows, budgets=BUDGETS)
if paired:
    df_paired = pd.DataFrame(paired)
    print('=== Paired Comparison (AB-MEDAL vs. MEDAL at Matched Search Budget) ===')
    display(df_paired[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']])


## 10. Interactive Search Tree & Decision Trace Inspector
Examine candidate shortlists, 2-step FIG rewards, and lookahead state entropy drops.

In [ ]:
import json

sample_ex_id = prepared[0]['example']['example_id']
chk_dir = OUTPUT_DIR / STAGE / 'ab_medal' / sample_ex_id / 'checkpoints/256'
if (chk_dir / 'metrics.json').is_file():
    met = json.loads((chk_dir / 'metrics.json').read_text())
    print(f'Sample {sample_ex_id} @ Budget 256:')
    print(f'  Selected Node: {met["selected_node_id"]} | Depth: {met["selected_depth"]} | Q: {met["selected_q"]:.4f}')
    print(f'  Exact Match: {met["exact_match"]} | Pred: {met["predicted_answer"]} | Ref: {met["reference_answer"]}')


## 11. Synchronize All Artifacts to Google Drive
Explicitly flush and sync all in-flight buffers to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed successfully.')
